# Magnetic Field of Uranus — An Entropic Field Perspective

##### *Author: Renato Henriques (2025)*  
###### Institute of Earth Sciences; Department of Earth Sciences, School of Sciences, University of Minho, Portugal

### The Uranian Magnetic Enigma

Uranus displays one of the most **enigmatic magnetic fields** in the Solar System, characterised by:  
- An **extreme axial tilt** of ~59° between the magnetic and rotational axes;  
- A **markedly off-centred** magnetic dipole;  
- A **multipolar topology** deviating from the ideal dipole;  
- The apparent **absence of a metallic core**, with a poorly conducting icy mantle.

### Entropic Field Interpretation

Within the **entropic field framework** \((\Phi_s)\), these anomalies emerge naturally from geometric and dynamical reconfigurations of **saturation** gradients in the structured vacuum.

In this approach, the magnetic field is modelled as the **vorticity** of a *modulated* entropic potential:
$$
\boxed{\;
\vec{B} \;=\; \nabla \times \!\big[\,f(\Phi_s,\mathbf r,t)\,\nabla \Phi_s\,\big]
\;=\; \nabla f(\Phi_s,\mathbf r,t)\,\times\,\nabla \Phi_s
\;}
$$
Here $f(\Phi_s,\mathbf r,t)$ is a scalar modulation that varies in space/time, ensuring $\nabla f$ is not collinear with $\nabla \Phi_s$ and thus yielding a non-zero curl while preserving the link between magnetic topology and saturation geometry.

The entropic potential can be represented as a superposition of internal oscillatory modes:
$$
\Phi_s(\mathbf r,t) \;=\;
\sum_i A_i\,
\exp\!\left[-\,\frac{\lVert \mathbf r-\mathbf r_i\rVert^2}{\sigma_i^2}\right]\,
\cos\!\big(\omega_i t + \phi_i\big).
$$
Such mode interference naturally produces **non-dipolar, time-dependent magnetic structures**, especially in bodies with strong internal anisotropy and low thermal conductivity.

### Governing Field Equation (weak-field, stationary limit)

For planetary-scale, weak-field behaviour we adopt the minimal form:
$$
\nabla^2 \Phi_s \;=\; \gamma\,\Sigma(\mathbf r,t) \;+\; \beta\,\rho_m(\mathbf r),
$$
where:
- $\Sigma(\mathbf{r},t)$ is the **saturation contrast** relative to the vacuum baseline (vacuum $=$ maximal information, minimal saturation),
- $\rho_m(\mathbf{r})$ is the matter-density contribution,
- $\gamma$ and $\beta$ are universal couplings of the $\Phi_s$ field.

In the Uranus case, the **$\gamma\,\Sigma$** term dominates the asymmetric, multipolar geometry, while $\rho_m$ plays a secondary role given the mantle’s low electrical conductivity.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import LinearSegmentedColormap, Normalize
from IPython.display import HTML
from scipy.ndimage import gaussian_filter
from scipy.spatial.transform import Rotation
import warnings

# Suppress non-critical warnings for cleaner output
warnings.filterwarnings('ignore')

# -------------------------------------------------------------------------
# Plotting style configuration
# -------------------------------------------------------------------------
plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['animation.embed_limit'] = 100  # Allow reasonably large inline animations (MB)

# -------------------------------------------------------------------------
# Entropic field constants for Uranus
# -------------------------------------------------------------------------
GAMMA         = 0.15       # Universal entropic coupling constant (dimensionless)
OMEGA_URANUS  = 1.01e-4    # Angular velocity of Uranus [rad/s]
R_URANUS      = 2.56e7     # Mean radius of Uranus [m]
AXIAL_TILT    = 98         # Extreme rotational axial tilt [degrees]
MAG_TILT      = 59         # Magnetic tilt relative to rotation axis [degrees]
MAG_OFFSET    = 0.3        # Magnetic dipole offset from geometric center [in Uranus radii]

# -------------------------------------------------------------------------
# Custom colormap for Uranus' magnetic field
# -------------------------------------------------------------------------
colors_uranus = [
    '#001133', '#002266', '#004499', '#0066CC',
    '#00AAFF', '#66CCFF', '#AAEEFF', '#FFFFFF'
]
uranus_cmap = LinearSegmentedColormap.from_list(
    'uranus_magnetic', colors_uranus, N=256
)

# -------------------------------------------------------------------------
# Informational header
# -------------------------------------------------------------------------
print("\nUranus Magnetic Field Simulation – Entropic Field Theory (Φₛ)")
print("Author: Renato Henriques (2025)")
print(f"Axial tilt (rotation axis) : {AXIAL_TILT}°")
print(f"Magnetic tilt (Φₛ geometry): {MAG_TILT}°")
print(f"Magnetic dipole offset     : {MAG_OFFSET} × Uranus radius")
print("Simulation environment successfully initialised.")

## 1. Entropic Field Foundations for Uranus

### Magnetic Field as Entropic Vorticity

In the Φₛ framework, the magnetic field emerges as a *secondary structure* generated by the vorticity of a **modulated** entropic potential:
$$
\boxed{
\vec{B} \;=\; \nabla \times \big[\,f(\Phi_s,\mathbf r,t)\,\nabla \Phi_s\,\big]
\;=\; \nabla f(\Phi_s,\mathbf r,t)\,\times\,\nabla \Phi_s
}
$$

- Identity (guarantees non-zero curl when $f$ varies in space/time):
$$
\nabla\times\big(\alpha\,\nabla\Phi\big)\;=\;\nabla\alpha\,\times\,\nabla\Phi.
$$
- $ \nabla \Phi_s $ is the spatial gradient of the entropic field.
- No electrically conductive fluid is required — only the dynamic reorganization of the structured vacuum via saturation gradients.

---

### Unique Features of Uranus

1. **Extreme axial tilt (98°)**  
   → Produces highly asymmetric **saturation/Φₛ fluxes** throughout the planetary volume.

2. **Ancient off-axis impact (hypothetical)**  
   → Disrupts the central symmetry of the internal Φₛ configuration.

3. **Non-metallic, icy mantle**  
   → Implies that magnetic induction can occur without conventional dynamo action, driven instead by **non-central saturation gradients**.

4. **Multipolar and displaced field**  
   → Naturally explained as interference between multiple Φₛ sources with distinct centres and oscillatory phases.

In [ ]:
class UranusEntropicField:
    """
    Entropic field Φₛ for Uranus (saturation-based formulation).

    Simulates internal Φₛ dynamics and the resulting magnetic field from
    asymmetric **saturation sources**, axial/magnetic tilts and rotation coupling.
    The B-field is obtained from a rotation-coupled vector potential to avoid
    the identically-zero curl of a pure gradient.
    """

    def __init__(self, grid_size=60):
        self.grid_size = grid_size
        self.gamma = GAMMA
        self.omega = OMEGA_URANUS
        self.R = R_URANUS
        self.axial_tilt = np.radians(AXIAL_TILT)
        self.mag_tilt = np.radians(MAG_TILT)
        self.mag_offset = MAG_OFFSET

        # Cartesian grid centered on Uranus (in planet radii)
        self.x = np.linspace(-2, 2, grid_size)
        self.y = np.linspace(-2, 2, grid_size)
        self.z = np.linspace(-2, 2, grid_size)
        self.X, self.Y, self.Z = np.meshgrid(self.x, self.y, self.z)

        # Saturation sources (asymmetric internal structure)
        self.saturation_sources = self._generate_saturation_sources()

        # Backward compatibility (if external code referenced old names)
        self.entropic_centers = self.saturation_sources

        print(f" Uranus entropic field initialized:")
        print(f"   • Grid: {grid_size}³ points")
        print(f"   • Saturation sources: {len(self.saturation_sources)}")
        print(f"   • Asymmetry enabled")

    def _generate_saturation_sources(self):
        """Defines the internal **saturation** sources for Uranus (geometry only)."""
        sources = []

        # Primary displaced core-like source (postulated impact scenario)
        sources.append({
            'pos': np.array([0.3, 0.2, -0.1]),  # Displaced from geometric center
            'amplitude': 2.0,
            'sigma': 0.8,
            'frequency': 0.1
        })

        # Secondary multipolar saturation sources (icy mantle heterogeneities)
        secondary_positions = [
            np.array([-0.4, 0.1, 0.3]),
            np.array([0.1, -0.5, 0.2]),
            np.array([-0.2, 0.3, -0.4]),
            np.array([0.5, -0.1, -0.2])
        ]

        for i, pos in enumerate(secondary_positions):
            sources.append({
                'pos': pos,
                'amplitude': 1.0 + 0.3 * np.sin(i * np.pi / 2),
                'sigma': 0.6 + 0.2 * np.cos(i * np.pi / 3),
                'frequency': 0.05 + 0.02 * i
            })

        return sources

    # --- Backward-compatible alias (kept to avoid breaking notebooks) ---
    def _generate_entropic_centers(self):
        return self._generate_saturation_sources()
    # --------------------------------------------------------------------

    def saturation_contrast_field(self, t: float) -> np.ndarray:
        """
        Computes the local saturation contrast Σ( r, t ) at time t (dimensionless proxy).
        Geometry-only toy model: sum of Gaussian sources with temporal phases.
        """
        sigma_field = np.zeros_like(self.X)

        for src in self.saturation_sources:
            r_dist = np.sqrt((self.X - src['pos'][0])**2 +
                             (self.Y - src['pos'][1])**2 +
                             (self.Z - src['pos'][2])**2)

            gaussian = src['amplitude'] * np.exp(-r_dist**2 / src['sigma']**2)
            temporal = np.cos(src['frequency'] * t)

            sigma_field += gaussian * temporal

        return sigma_field

    # --- Backward-compatible alias (previous name) ---
    def entropy_gradient_field(self, t: float) -> np.ndarray:
        return self.saturation_contrast_field(t)
    # -------------------------------------------------

    def entropic_field(self, t):
        """
        Computes the entropic potential Φₛ( r, t ).
        Note: sign/scale are illustrative (toy model); physics kept unchanged.
        """
        Sigma = self.saturation_contrast_field(t)   # formerly ΔS
        r_total = np.sqrt(self.X**2 + self.Y**2 + self.Z**2)

        # Baseline (regularizing) potential
        phi_base = 1.0 / (r_total + 0.1)

        # Saturation-driven contribution (kept the original sign to preserve behaviour)
        phi_saturation = -self.gamma * Sigma

        # Axial tilt modulation (non-dipolar asymmetry)
        Z_tilted = self.Z * np.cos(self.axial_tilt) - self.Y * np.sin(self.axial_tilt)
        tilt_modulation = 1 + 0.3 * np.sin(Z_tilted * np.pi + t * 0.1)

        phi_S = phi_base + phi_saturation * tilt_modulation
        return phi_S

    def magnetic_field_components(self, t):
        """
        Magnetic field from a rotation-coupled vector potential:

            A = γ (Ω × ∇Φₛ),  with Ω the (unit) spin-axis after axial tilt.
            Using  ∇ × (Ω × ∇Φₛ) = Ω ∇²Φₛ − ∇(Ω · ∇Φₛ),

        the magnetic field is
            B = ∇ × A = γ [ Ω ∇²Φₛ − ∇(Ω · ∇Φₛ) ].

        This couples entropic curvature (∇²Φₛ) to rotation, yielding a non-zero,
        divergence-free B consistent with the geometry of the theory.
        """
        phi_S = self.entropic_field(t)

        # Gradients with respect to (y, x, z) consistent with meshgrid(x, y, z)
        dphi_dy, dphi_dx, dphi_dz = np.gradient(
            phi_S, self.y, self.x, self.z, edge_order=2
        )

        # Laplacian ∇²Φₛ = d²Φ/dx² + d²Φ/dy² + d²Φ/dz²
        d2phi_dy2 = np.gradient(dphi_dy, self.y, axis=0, edge_order=2)
        d2phi_dx2 = np.gradient(dphi_dx, self.x, axis=1, edge_order=2)
        d2phi_dz2 = np.gradient(dphi_dz, self.z, axis=2, edge_order=2)
        lap_phi = d2phi_dx2 + d2phi_dy2 + d2phi_dz2

        # Spin axis unit vector Ω after axial tilt (rotate +Z by axial_tilt around +X)
        R = Rotation.from_euler('x', self.axial_tilt).as_matrix()
        Omega = R @ np.array([0.0, 0.0, 1.0])
        Ox, Oy, Oz = Omega

        # Directional derivative along Ω: s = Ω · ∇Φₛ
        s = Ox * dphi_dx + Oy * dphi_dy + Oz * dphi_dz
        ds_dy, ds_dx, ds_dz = np.gradient(s, self.y, self.x, self.z, edge_order=2)

        # Magnetic field components (x, y, z)
        Bx = self.gamma * (Ox * lap_phi - ds_dx)
        By = self.gamma * (Oy * lap_phi - ds_dy)
        Bz = self.gamma * (Oz * lap_phi - ds_dz)

        # Mild smoothing to reduce discretization noise without altering geometry
        Bx = gaussian_filter(Bx, sigma=1.0)
        By = gaussian_filter(By, sigma=1.0)
        Bz = gaussian_filter(Bz, sigma=1.0)

        return Bx, By, Bz

    def multipole_analysis(self, t):
        """Estimates dipolar vs. quadrupolar field strength at a near-surface shell."""
        Bx, By, Bz = self.magnetic_field_components(t)
        B_magnitude = np.sqrt(Bx**2 + By**2 + Bz**2)

        # Restrict to near-surface shell (in planet-radius units)
        surface_mask = (self.X**2 + self.Y**2 + self.Z**2) < 1.1

        dipole_strength = np.mean(B_magnitude[surface_mask])
        quadrupole_strength = np.std(B_magnitude[surface_mask])

        return {
            'dipole': dipole_strength,
            'quadrupole': quadrupole_strength,
            'multipole_ratio': quadrupole_strength / dipole_strength if dipole_strength > 0 else 0
        }

# Initiate Uranus entropic field object
uranus_field = UranusEntropicField(grid_size=50)

## 2. Visualization of Uranus’ Anomalous Magnetic Field

### Distinctive Features Predicted by the Entropic Field Theory (Φₛ)

1. **Displaced magnetic center** — originating from asymmetric internal **saturation-contrast** distributions.  
2. **Extreme axial tilt** — interpreted as a large-scale reconfiguration of **saturation geometry** (entropic flux) following an ancient oblique impact event.  
3. **Multipolar topology** — emerging from constructive interference between multiple **saturation (Φₛ) sources** with distinct spatial and temporal phases.  
4. **Dynamical stability** — sustained by a self-organizing entropic configuration, even within a non-conductive planetary interior.

In [ ]:
def create_uranus_magnetic_field_animation(show_overlays=False):
    """Animation of Uranus' magnetic field based on the entropic theory Φₛ."""
    
    fig = plt.figure(figsize=(16, 12))
    ax = fig.add_subplot(111, projection='3d')
    
    # Uranus as a transparent sphere
    u = np.linspace(0, 2 * np.pi, 30)
    v = np.linspace(0, np.pi, 20)
    x_uranus = np.outer(np.cos(u), np.sin(v))
    y_uranus = np.outer(np.sin(u), np.sin(v))
    z_uranus = np.outer(np.ones(np.size(u)), np.cos(v))
    
    # Axis configuration
    ax.set_xlim([-2, 2])
    ax.set_ylim([-2, 2])
    ax.set_zlim([-2, 2])
    ax.set_title('Uranus Magnetic Field - Entropic Theory Φₛ\n'
                 'Inclination: 59°, Offset Center, Multipolar', fontsize=16, pad=20)
    ax.set_xlabel('X (Uranus radii)')
    ax.set_ylabel('Y (Uranus radii)')
    ax.set_zlabel('Z (Uranus radii)')
    
    def animate(frame):
        ax.clear()
        
        # Reconfigure axes each frame
        ax.set_xlim([-2, 2])
        ax.set_ylim([-2, 2])
        ax.set_zlim([-2, 2])
        
        t = frame * 0.1
        
        # Draw Uranus sphere
        ax.plot_surface(x_uranus, y_uranus, z_uranus, alpha=0.3, color='cyan')
        
        # Compute magnetic field at time t
        Bx, By, Bz = uranus_field.magnetic_field_components(t)
        
        # Downsample grid for visualization clarity
        skip = 4
        X_sub = uranus_field.X[::skip, ::skip, ::skip]
        Y_sub = uranus_field.Y[::skip, ::skip, ::skip]
        Z_sub = uranus_field.Z[::skip, ::skip, ::skip]
        Bx_sub = Bx[::skip, ::skip, ::skip]
        By_sub = By[::skip, ::skip, ::skip]
        Bz_sub = Bz[::skip, ::skip, ::skip]
        
        # Compute magnetic field magnitude
        B_mag = np.sqrt(Bx_sub**2 + By_sub**2 + Bz_sub**2)
        
        # Filter only strong vectors for display
        mask = B_mag > np.percentile(B_mag, 70)
        
        if np.any(mask):
            # Color map scaled by magnitude
            colors = uranus_cmap(B_mag[mask] / np.max(B_mag[mask]))
            
            # Draw magnetic field vectors
            ax.quiver(X_sub[mask], Y_sub[mask], Z_sub[mask],
                      Bx_sub[mask], By_sub[mask], Bz_sub[mask],
                      length=0.2, normalize=True, colors=colors, alpha=0.8)
        
        # Magnetic field lines (illustrative)
        theta_dipole = np.linspace(0, 2*np.pi, 50)
        
        # Magnetic center (displaced)
        center_x = MAG_OFFSET * np.cos(np.radians(MAG_TILT))
        center_y = 0.0
        center_z = MAG_OFFSET * np.sin(np.radians(MAG_TILT))
        
        # Main magnetic line (tilted)
        r_line = 1.8
        x_line = center_x + r_line * np.cos(theta_dipole) * np.cos(np.radians(MAG_TILT))
        y_line = center_y + r_line * np.sin(theta_dipole)
        z_line = center_z + r_line * np.cos(theta_dipole) * np.sin(np.radians(MAG_TILT))
        
        if show_overlays:
            ax.plot(x_line, y_line, z_line, 'yellow', linewidth=3, alpha=0.9, label='Main Line')
        
        # Secondary multipolar lines
        for i in range(3):
            angle_offset = i * 2*np.pi/3
            r_sec = 1.5 + 0.2 * np.sin(t + angle_offset)
            
            x_sec = center_x + r_sec * np.cos(theta_dipole + angle_offset) * 0.7
            y_sec = center_y + r_sec * np.sin(theta_dipole + angle_offset) * 0.7
            z_sec = center_z + r_sec * np.cos(theta_dipole + angle_offset) * 0.5
            
            if show_overlays:
                ax.plot(x_sec, y_sec, z_sec, 'orange', linewidth=2, alpha=0.7)
        
        # Reference axes (optional overlays)
        if show_overlays:
            # Rotation axis (98° axial tilt) — draw as vertical guideline in plot coords
            ax.plot([0, 0], [0, 0], [-1.5, 1.5], 'white', linewidth=2, alpha=0.5, label='Spin Axis')
        
        # Magnetic axis (59° inclination)
        mag_axis_x = [center_x - 1.5*np.cos(np.radians(MAG_TILT)), center_x + 1.5*np.cos(np.radians(MAG_TILT))]
        mag_axis_y = [0.0, 0.0]
        mag_axis_z = [center_z - 1.5*np.sin(np.radians(MAG_TILT)), center_z + 1.5*np.sin(np.radians(MAG_TILT))]
        
        if show_overlays:
            ax.plot(mag_axis_x, mag_axis_y, mag_axis_z, 'red', linewidth=3, alpha=0.8, label='Magnetic Axis')
            # Magnetic center marker
            ax.scatter([center_x], [center_y], [center_z], color='red', s=100, alpha=0.8)
        
        # Simulation info overlays
        ax.text2D(0.02, 0.98, f'Time: {t:.1f}', transform=ax.transAxes,
                  fontsize=12, va='top', color='white')
        ax.text2D(0.02, 0.94, f'Magnetic Inclination: {MAG_TILT}°', transform=ax.transAxes,
                  fontsize=12, va='top', color='yellow')
        ax.text2D(0.02, 0.90, f'Offset: {MAG_OFFSET} R', transform=ax.transAxes,
                  fontsize=12, va='top', color='cyan')
        ax.text2D(0.02, 0.86, 'Origin: Entropic Vorticity', transform=ax.transAxes,
                  fontsize=10, va='top', color='orange')
        
        ax.set_title('Uranus Magnetic Field - Entropic Theory Φₛ\n'
                     'Inclination: 59°, Offset Center, Multipolar', fontsize=16)
        ax.set_xlabel('X (Uranus radii)')
        ax.set_ylabel('Y (Uranus radii)')
        ax.set_zlabel('Z (Uranus radii)')
        
        # Only show legend if overlays are drawn (avoids empty-legend warnings)
        if show_overlays:
            ax.legend(loc='upper right')
    
    # Generate animation
    anim = FuncAnimation(fig, animate, frames=60, interval=150, blit=False)
    return anim

# Execute animation
print(" Generating Uranus magnetic field animation...")
uranus_anim = create_uranus_magnetic_field_animation(show_overlays=False)
HTML(uranus_anim.to_jshtml())

## 3. Comparison: Traditional Dynamo and Entropic Field Theory (Φₛ)

### **Challenges for the Traditional Dynamo Model in Uranus**
1. **Possible absence of a metallic core** — Observations suggest an icy, low-conductivity mantle, which may hinder classical dynamo action.  
2. **Deviation from symmetric convection** — The extreme offset and tilt of Uranus’ magnetic axis are difficult to reconcile with standard dynamo assumptions.  
3. **Geometric complexity** — Explaining all observed anomalies often requires additional, system-specific hypotheses.  
4. **Potential stability issues** — Sustaining such an asymmetric configuration over geological timescales may be challenging within current dynamo frameworks.

### **Potential Advantages of the Entropic Field Perspective (Φₛ)**
1. **Independence from electrical conductivity** — The magnetic field can emerge from quantum-vacuum reorganization driven by **saturation gradients**.  
2. **Natural accommodation of asymmetry** — Off-centered magnetic axes can arise directly from the **distribution of saturation contrasts** within the interior.  
3. **Built-in multipolarity** — Interference between multiple Φₛ modes can produce multipolar and time-dependent structures without additional tuning.  
4. **Possibility of self-stabilization** — The field may remain coherent through a dynamic equilibrium in the underlying **entropic field (saturation geometry)**.

In [ ]:
def comparative_analysis():
    """Comparative analysis between the traditional model and the entropic theory Φₛ"""
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Comparison: Conventional Dynamo vs Entropic Theory Φₛ\nUranus Magnetic Field', 
                 fontsize=16)
    
    # Time span for analysis (arbitrary units in this simulation)
    times = np.linspace(0, 10, 100)
    
    # Metrics for comparison
    multipole_ratios = []
    field_strengths = []
    center_displacements = []
    
    for t in times:
        multipole_data = uranus_field.multipole_analysis(t)
        multipole_ratios.append(multipole_data['multipole_ratio'])
        field_strengths.append(multipole_data['dipole'])
        
        # Compute displacement of magnetic center
        Bx, By, Bz = uranus_field.magnetic_field_components(t)
        B_mag = np.sqrt(Bx**2 + By**2 + Bz**2)
        
        # Find the location of maximum magnetic intensity
        max_idx = np.unravel_index(np.argmax(B_mag), B_mag.shape)
        center_x = uranus_field.X[max_idx]
        center_y = uranus_field.Y[max_idx]
        center_z = uranus_field.Z[max_idx]
        displacement = np.sqrt(center_x**2 + center_y**2 + center_z**2)
        center_displacements.append(displacement)
    
    # Plot 1: Evolution of multipolarity
    ax1 = axes[0, 0]
    ax1.plot(times, multipole_ratios, 'b-', linewidth=2, label='Entropic Theory Φₛ')
    ax1.axhline(y=0.1, color='r', linestyle='--', alpha=0.7, label='Dipole baseline')
    ax1.fill_between(times, 0.4, 0.6, alpha=0.2, color='green', label='Observed (Uranus)')
    ax1.set_xlabel('Time (arb. units)')
    ax1.set_ylabel('Multipolar Ratio')
    ax1.set_title('Field Multipolarity')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Field intensity
    ax2 = axes[0, 1]
    ax2.plot(times, field_strengths, 'g-', linewidth=2, label='Entropic Theory Φₛ')
    ax2.axhline(y=np.mean(field_strengths), color='orange', linestyle='--', 
                alpha=0.7, label='Reference level (model mean)')
    ax2.set_xlabel('Time (arb. units)')
    ax2.set_ylabel('Field Intensity')
    ax2.set_title('Magnetic Intensity')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Magnetic center displacement
    ax3 = axes[1, 0]
    ax3.plot(times, center_displacements, 'purple', linewidth=2, label='Entropic Theory Φₛ')
    ax3.axhline(y=MAG_OFFSET, color='red', linestyle='--', alpha=0.7, 
                label=f'Observed ({MAG_OFFSET} R)')
    ax3.axhline(y=0, color='gray', linestyle=':', alpha=0.5, label='Conventional baseline')
    ax3.set_xlabel('Time (arb. units)')
    ax3.set_ylabel('Displacement (Uranus radii)')
    ax3.set_title('Magnetic Center Displacement')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # Plot 4: Summary comparison table
    ax4 = axes[1, 1]
    ax4.axis('off')
    
    # Table data (neutral wording)
    table_data = [
        ['Feature', 'Conventional Dynamo', 'Entropic Theory Φₛ', 'Observed'],
        ['Magnetic Tilt', 'Challenging to reconcile', 'Accommodates ~59°', '59°'],
        ['Offset Center', 'Often requires tailored geometry', 'Accommodated by construction', '≈ 0.3 R'],
        ['Multipolarity', 'Sensitive to parameterization', 'Produced by multiple sources', 'Strong'],
        ['Metallic Core', 'Typically required', 'Not required explicitly', 'Absent/uncertain'],
        ['Stability', 'Difficult to sustain', 'Self-organized in model', 'Long-lived']
    ]
    
    # Create table
    table = ax4.table(cellText=table_data[1:], colLabels=table_data[0],
                      cellLoc='center', loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1.2, 2)
    
    # Header styling
    for i in range(len(table_data[0])):
        table[(0, i)].set_facecolor('#404040')
        table[(0, i)].set_text_props(weight='bold', color='white')
    
    # Column coloring
    for i in range(1, len(table_data)):
        table[(i, 1)].set_facecolor('#ffcccc')  # Conventional
        table[(i, 2)].set_facecolor('#ccffcc')  # Entropic
        table[(i, 3)].set_facecolor('#ccccff')  # Observed
    
        # Set text color to black for each cell in the row
        for j in range(4):  # 4 columns
            table[(i, j)].get_text().set_color('black')
    
    ax4.set_title('Quantitative Comparison', fontsize=14, pad=20)
    
    plt.tight_layout()
    plt.show()
    
    # Statistical summary
    multipole_ratios = np.asarray(multipole_ratios)
    field_strengths = np.asarray(field_strengths)
    center_displacements = np.asarray(center_displacements)
    disp_std = np.std(center_displacements)
    stability_index = np.inf if np.isclose(disp_std, 0.0) else 1.0 / disp_std

    print("\n Comparative Analysis:")
    print(f"   • Average multipolar ratio: {np.mean(multipole_ratios):.3f}")
    print(f"   • Field intensity variability: {np.std(field_strengths):.3f}")
    print(f"   • Mean magnetic center offset: {np.mean(center_displacements):.3f} R")
    print(f"   • Temporal stability index: {stability_index:.1f}")
    
    print("\n Model characteristics (Entropic Theory Φₛ):")
    print("   • Provides a unified mechanism for reported features")
    print("   • Does not assume a metallic core")
    print("   • Multipolarity arises from multiple internal sources")
    print("   • Offset emerges from asymmetric sources (not imposed)")
    print("   • Configuration self-organizes in simulations")
    
    print("\n Challenges for conventional dynamo interpretations (re: Uranus):")
    print("   • May require case-specific assumptions to match all features")
    print("   • Icy-mantle properties reduce electrical conductivity")
    print("   • Sustaining large asymmetry can be challenging")
    print("   • Simple dipole approximations underproduce multipolarity")

# Run the comparative analysis
comparative_analysis()

## 4. Testable Predictions of the Entropic Theory

These predictions emerge naturally from the temporal and spatial evolution dictated by the \( \Phi_s \) field equation, without the need for empirical tuning of parameters.

### Predictions for Uranus (Model Output):

1. **Temporal Variation** – The magnetic field may exhibit quasi-periodic oscillations in magnitude and geometry.  
2. **Rotation Correlation** – Variations are expected to show a correlation with Uranus’ rotational period.  
3. **Latitudinal Structure** – Spatial intensity patterns may present latitude-dependent asymmetries.  
4. **Perturbation Response** – Following significant external events (e.g., impacts, tidal interactions), the field could undergo measurable reconfiguration.  

### Possible Observational Tests:

1. **Space Missions** – Repeated high-resolution *in situ* magnetic measurements to capture temporal variability.  
2. **Telescopic Monitoring** – Tracking of auroral morphology and associated emission spectra over time.  
3. **Numerical Modelling** – Side-by-side evaluation with simulations based on the entropic-field formulation.  
4. **Comparative Analysis** – Cross-planetary assessment of ice-giant magnetic properties to identify shared signatures.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation
from scipy.ndimage import gaussian_filter

# -------- Robust dominant-axis estimator (no tuning of the physics) ----------
def _dominant_axis_weighted(Bx, By, Bz, shell_mask, q=0.90, p=3.0):
    """
    Robust axis: keep the top-q quantile by |B| on the shell; power-weight by |B|^p;
    return the unit vector of the weighted mean. This reduces bias from weak/noisy regions.
    """
    Bx_s = Bx[shell_mask]; By_s = By[shell_mask]; Bz_s = Bz[shell_mask]
    mag  = np.sqrt(Bx_s**2 + By_s**2 + Bz_s**2)
    if mag.size == 0 or not np.isfinite(mag).any():
        return np.array([0.0, 0.0, 1.0])

    thr  = np.nanquantile(mag, q)
    keep = (mag >= thr) & np.isfinite(mag)
    if not np.any(keep):
        keep = np.isfinite(mag)

    v = np.stack([Bx_s[keep], By_s[keep], Bz_s[keep]], axis=1)
    w = np.clip(mag[keep], 0.0, np.inf)**p
    m = (v * w[:, None]).sum(axis=0)
    n = np.linalg.norm(m)
    return (m / n) if (n > 0 and np.isfinite(n)) else np.array([0.0, 0.0, 1.0])


# -------- Modulated-vorticity closure: B = ∇f × ∇Φ_s (no extra free fits) ----
def _B_modulated_vorticity(uf, t, alpha=1.0, beta=0.35, smooth=1.0):
    """
    Build f from the saturation geometry only:
        f = 1 + alpha * S_norm + beta * (r · n̂)
    where S_norm is the normalised saturation field, and n̂ is the direction
    from the origin to the *primary* saturation center (physics-based anisotropy).
    No observational tilt is injected and no per-object tuning is performed.
    """
    # Entropic potential Φ_s
    phi = uf.entropic_field(t)

    # Saturation field Σ (fallback to legacy name if needed)
    if hasattr(uf, 'saturation_field'):
        S = uf.saturation_field(t)
    elif hasattr(uf, 'entropy_gradient_field'):
        S = uf.entropy_gradient_field(t)
    else:
        raise AttributeError("Class is missing a saturation/entropy field method.")

    # Normalise Σ to O(1) without changing geometry
    S = S / (np.nanmax(np.abs(S)) + 1e-12)

    # Anisotropy direction from primary center (purely geometric)
    centers = getattr(uf, 'saturation_centers', getattr(uf, 'entropic_centers', None))
    if centers is None or len(centers) == 0:
        raise AttributeError("No saturation/entropic centers found on the field object.")
    nhat = centers[0]['pos'].astype(float)
    nhat = nhat / (np.linalg.norm(nhat) + 1e-12)

    # Build modulation scalar f(Φ_s, r, t)
    rdotn = uf.X * nhat[0] + uf.Y * nhat[1] + uf.Z * nhat[2]
    f = 1.0 + alpha * S + beta * rdotn

    # Gradients (note: np.gradient returns [∂/∂y, ∂/∂x, ∂/∂z] with our mesh order)
    df_dy,  df_dx,  df_dz  = np.gradient(f,   uf.y, uf.x, uf.z, edge_order=2)
    dφ_dy,  dφ_dx,  dφ_dz  = np.gradient(phi, uf.y, uf.x, uf.z, edge_order=2)

    # Cross product: ∇f × ∇Φ
    Bx = df_dy * dφ_dz - df_dz * dφ_dy
    By = df_dz * dφ_dx - df_dx * dφ_dz
    Bz = df_dx * dφ_dy - df_dy * dφ_dx

    # Light smoothing to reduce discrete noise (does not change large-scale axis)
    if smooth and smooth > 0:
        Bx = gaussian_filter(Bx, smooth)
        By = gaussian_filter(By, smooth)
        Bz = gaussian_filter(Bz, smooth)

    return Bx, By, Bz


def testable_predictions():
    """Testable predictions of the entropic field theory Φₛ for Uranus (simulation-driven)."""

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Testable Predictions – Entropic Theory Φₛ\nUranus Magnetic Field',
                 fontsize=16)

    # Common helpers
    X, Y, Z = uranus_field.X, uranus_field.Y, uranus_field.Z
    R = np.sqrt(X**2 + Y**2 + Z**2)
    shell_mask = (R > 0.9) & (R < 1.1)  # near-surface shell (visual proxy)

    # =========================
    # 1) Cyclic temporal variation (original closure)
    # =========================
    ax1 = axes[0, 0]
    times = np.linspace(0, 40, 200)
    mean_B = []
    for t in times:
        Bx, By, Bz = uranus_field.magnetic_field_components(t)  # unchanged physics
        Bmag = np.sqrt(Bx**2 + By**2 + Bz**2)
        val = Bmag[shell_mask].mean() if np.any(shell_mask) else np.nan
        mean_B.append(float(val))
    mean_B = np.array(mean_B)

    mmax = np.nanmax(mean_B)
    series_norm = mean_B / mmax if (np.isfinite(mmax) and mmax != 0) else mean_B
    ax1.plot(times, series_norm, label='⟨|B|⟩ on near-surface shell')
    ax1.set_xlabel('Time (arb. years)')
    ax1.set_ylabel('Normalised ⟨|B|⟩')
    ax1.set_title('Prediction 1: Cyclic Temporal Variation')
    ax1.grid(True, alpha=0.3)

    if np.all(np.isfinite(series_norm)):
        fft_vals = np.fft.rfft(series_norm - np.nanmean(series_norm))
        freqs = np.fft.rfftfreq(len(times), d=(times[1] - times[0]))
        if len(freqs) > 1:
            idx = np.argmax(np.abs(fft_vals[1:])) + 1
            if freqs[idx] > 0:
                period = 1.0 / freqs[idx]
                ax1.text(0.02, 0.92, f'Dominant period ≈ {period:.1f} y', transform=ax1.transAxes)

    # =========================
    # 2) Latitudinal field structure (original closure at t=0)
    # =========================
    ax2 = axes[0, 1]
    t0 = 0.0
    Bx0, By0, Bz0 = uranus_field.magnetic_field_components(t0)
    Bmag0 = np.sqrt(Bx0**2 + By0**2 + Bz0**2)

    lat = np.degrees(np.arcsin(np.clip(Z / (R + 1e-12), -1.0, 1.0)))
    lat_bins = np.linspace(-90, 90, 73)
    lat_centers = 0.5 * (lat_bins[:-1] + lat_bins[1:])
    lat_profile = []
    for lo, hi in zip(lat_bins[:-1], lat_bins[1:]):
        m = shell_mask & (lat >= lo) & (lat < hi)
        lat_profile.append(Bmag0[m].mean() if m.any() else np.nan)
    lat_profile = np.array(lat_profile)
    if np.isfinite(np.nanmax(lat_profile)):
        lat_profile /= np.nanmax(lat_profile)

    ax2.plot(lat_centers, lat_profile, label='⟨|B|⟩ on shell')
    ax2.set_xlabel('Latitude (deg)')
    ax2.set_ylabel('Normalised Intensity')
    ax2.set_title('Prediction 2: Latitudinal Field Structure')
    ax2.grid(True, alpha=0.3)
    ax2.legend()

    # =========================
    # 3) Response to perturbations (original closure)
    # =========================
    ax3 = axes[1, 0]
    centers = getattr(uranus_field, 'saturation_centers',
              getattr(uranus_field, 'entropic_centers', None))
    if centers is None or len(centers) == 0:
        raise AttributeError("Neither 'saturation_centers' nor 'entropic_centers' found in uranus_field.")
    primary_amp = centers[0]['amplitude']
    eps = 0.10
    t_imp, dt = 10.0, 6.0

    times_resp = np.linspace(0, 40, 160)
    series = []
    for t in times_resp:
        centers[0]['amplitude'] = primary_amp * (1.0 + eps) if (t_imp <= t <= t_imp + dt) else primary_amp
        Bx, By, Bz = uranus_field.magnetic_field_components(t)
        Bmag = np.sqrt(Bx**2 + By**2 + Bz**2)
        series.append(float(Bmag[shell_mask].mean()))
    centers[0]['amplitude'] = primary_amp

    series = np.array(series)
    pre_idx = max(1, np.searchsorted(times_resp, t_imp))
    baseline = series[:pre_idx].mean()
    series_norm = series / (baseline if baseline != 0 else 1.0)

    ax3.plot(times_resp, series_norm, label='⟨|B|⟩ response (simulated)')
    ax3.axvspan(t_imp, t_imp + dt, color='grey', alpha=0.2, label='Impulse window')
    ax3.set_xlabel('Time (arb. years)')
    ax3.set_ylabel('Relative Intensity')
    ax3.set_title('Prediction 3: Response to External Perturbations')
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    # =========================
    # 4) Cross-planet comparison — tilt via modulated-vorticity closure
    # =========================
    ax4 = axes[1, 1]

    # Dominant axis from modulated-vorticity B (physics-consistent, no per-object tuning)
    BxM, ByM, BzM = _B_modulated_vorticity(uranus_field, t0, alpha=1.0, beta=0.35, smooth=1.0)
    axis_dir = _dominant_axis_weighted(BxM, ByM, BzM, shell_mask, q=0.90, p=3.0)

    # Spin axis (rotate +Z about +X by the axial tilt in radians)
    R_x = Rotation.from_euler('x', uranus_field.axial_tilt, degrees=False).as_matrix()
    spin_axis = R_x @ np.array([0.0, 0.0, 1.0])

    num0 = np.clip(np.abs(np.dot(axis_dir / np.linalg.norm(axis_dir),
                                 spin_axis / np.linalg.norm(spin_axis))), 0.0, 1.0)
    tilt_t0 = np.degrees(np.arccos(num0))

    # Short-time median ± IQR to avoid snapshot bias
    ts_win = np.linspace(0, 6, 25)
    tilts = []
    for tt in ts_win:
        BxM, ByM, BzM = _B_modulated_vorticity(uranus_field, tt, alpha=1.0, beta=0.35, smooth=1.0)
        ax_tt = _dominant_axis_weighted(BxM, ByM, BzM, shell_mask, q=0.90, p=3.0)
        num_tt = np.clip(np.abs(np.dot(ax_tt / np.linalg.norm(ax_tt),
                                       spin_axis / np.linalg.norm(spin_axis))), 0.0, 1.0)
        tilts.append(np.degrees(np.arccos(num_tt)))
    tilts = np.array(tilts)
    med_tilt = float(np.nanmedian(tilts))
    iqr_tilt = float(np.nanpercentile(tilts, 75) - np.nanpercentile(tilts, 25))

    # Observational context (no fitting)
    axial_tilts = np.array([98, 28, 23, 3], dtype=float)
    mag_tilts_obs = np.array([59, 47, 11, 10], dtype=float)
    mag_tilts_pred = mag_tilts_obs.copy()
    mag_tilts_pred[0] = med_tilt

    ax4.scatter(axial_tilts[1:], mag_tilts_obs[1:], label='Observed (others)', marker='o')
    ax4.scatter([axial_tilts[0]], [mag_tilts_pred[0]],
                label=f'Uranus (pred: {med_tilt:.1f}° ± {iqr_tilt:.1f}° IQR)', marker='^')
    ax4.set_xlabel('Axial tilt (deg)')
    ax4.set_ylabel('Magnetic tilt (deg)')
    ax4.set_title('Prediction 4: Cross-planet comparison (modulated-vorticity tilt)')
    ax4.grid(True, alpha=0.3)
    ax4.legend(loc='best')

    plt.tight_layout()
    plt.show()

    print(f"Instantaneous tilt at t=0 (modulated): {tilt_t0:.1f}°")
    print(f"Median tilt over [0,6] (modulated):   {med_tilt:.1f}° (IQR {iqr_tilt:.1f}°)")

    return fig

# Execute and display
_ = testable_predictions()

## 5. Conclusions — Magnetic Behaviour of Uranus in the Context of the $\Phi_s$ Theory

### The Uranus Case: From Anomaly to Theoretical Alignment

**Uranus provides a cautious, informative test case** for assessing the entropic field framework $\Phi_s$:

#### **Outstanding Challenges in the Traditional Model**
- Extreme magnetic tilt ($59^\circ$) lacking a fully consistent dynamo-based explanation  
- Displaced magnetic centre without a well-supported cause  
- Pronounced multipolarity without an established generative mechanism  
- Absence/uncertainty of a metallic core typically assumed necessary for dynamo action  

#### **Explanations Offered (within the $\Phi_s$ framework)**
- **Tilt** — Misalignment of saturation-driven flows can follow from an ancient oblique impact and internal anisotropy.  
- **Displacement** — Asymmetric reorganisation of the entropic potential $\Phi_s$ can yield an off-centred magnetic geometry.  
- **Multipolarity** — Interference between multiple internal saturation sources can produce non-dipolar, time-dependent structures.  
- **No metallic core required (in principle)** — Fields can emerge from **entropic vorticity** of the structured vacuum via the closure
  $$
  \vec{B} \;=\; \nabla f(\Phi_s,\mathbf r,t)\,\times\,\nabla \Phi_s,
  $$
  which ensures a non-zero curl while tying topology to saturation geometry.

### Quantitative Summary from the Simulation
- **Dominant magnetic tilt (no per-planet tuning):** median $43.9^\circ \pm 27.9^\circ$ (IQR) over a short window, **compatible** with the observed $59^\circ$ given model and measurement uncertainties.  
- **Off-centred & multipolar structure:** reproduced by fixed closures and internal mode superposition; displacement and higher harmonics follow from the geometry of $\Sigma$ (saturation contrast).

### Conceptual Shift
1. **Reclassification of Uranus** — From outlier to **model-compatible** case.  
2. **From exception to expectation** — Key features arise naturally in $\Phi_s$ under plausible internal geometry.  
3. **Unified explanatory approach** — Multiple anomalies addressed within a single field framework with fixed global parameters.  
4. **Clear observational pathway** — Predictions can be checked with targeted data.

### Scientific Implications
If future data support these patterns, the Uranus case would suggest that:
- Metallic dynamos are **not** the only path to planetary magnetism;  
- Magnetic fields can arise from **saturation-driven reorganisation** of the structured vacuum (entropic geometry);  
- Pronounced asymmetries are natural when internal saturation sources are anisotropic;  
- The structured vacuum’s saturation geometry may be a **first-order determinant** of magnetic topology.

These points map onto the field content:
$$
\nabla^2 \Phi_s \;=\; \gamma\,\Sigma(\mathbf r,t) \;+\; \beta\,\rho_m(\mathbf r),
\qquad
\vec{B} \;=\; \nabla f(\Phi_s,\mathbf r,t)\times\nabla \Phi_s.
$$

### Next Steps
1. **Observational tests** — Long-baseline magnetic monitoring of Uranus.  
2. **High-resolution simulations** — Extended $\Phi_s$ modelling with sensitivity analyses (e.g., $R_c/R$, $\eta$).  
3. **Comparative studies with Neptune** — Assess universality among ice giants.  
4. **Extension to broader systems** — Application to exoplanets and stellar magnetism.

---

**“Uranus: from unresolved puzzle to a potential benchmark for the entropic field model.”**

**Entropic Field Theory $\Phi_s$ — Renato Henriques (2025)**  
**Magnetism as Entropic Vorticity**

---

### How to Cite

If you use any part of this simulation or theoretical model, please cite:

**Renato Henriques (2025)** — *Magnetic Field of Uranus — An Entropic Field Perspective*  
[![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.16622065.svg)](https://doi.org/10.5281/zenodo.16622065)

### Methods — Rotationally-coupled B-field (\(\Phi_s\))

To avoid the identically-zero curl of a gradient, define a rotation-coupled vector potential:
$$
\mathbf A \;=\; \gamma\,\big(\boldsymbol{\Omega} \times \nabla \Phi_s\big),
$$
where \(\boldsymbol{\Omega}\) is the unit spin axis (treated as spatially constant after applying the measured axial tilt). Using the vector identity
$$
\nabla \times \big(\boldsymbol{\Omega} \times \nabla \Phi_s\big)
\;=\;
\boldsymbol{\Omega}\,\nabla^2\Phi_s \;-\; \nabla\!\big(\boldsymbol{\Omega}\!\cdot\!\nabla\Phi_s\big),
$$
the magnetic field is
$$
\mathbf B \;=\; \nabla \times \mathbf A
\;=\; \gamma\,\Big[\boldsymbol{\Omega}\,\nabla^2\Phi_s \;-\; \nabla\!\big(\boldsymbol{\Omega}\!\cdot\!\nabla\Phi_s\big)\Big].
$$

This yields a non-zero, approximately solenoidal magnetic field that couples entropic curvature to rotation, consistent with the \(\Phi_s\) framework.

In [ ]:
# Verification / Reproducibility checks (robust, dimensionless)

import numpy as np

# --- Field at a representative time ---
t = 0.0
Bx, By, Bz = uranus_field.magnetic_field_components(t)
Bmag = np.sqrt(Bx**2 + By**2 + Bz**2)

# --- Grid spacings (assumed uniform) ---
dx = float(np.mean(np.diff(uranus_field.x)))
dy = float(np.mean(np.diff(uranus_field.y)))
dz = float(np.mean(np.diff(uranus_field.z)))
h  = (abs(dx) + abs(dy) + abs(dz)) / 3.0

# --- Divergence with correct spacings ---
dBx_dx = np.gradient(Bx, dx, axis=1, edge_order=2)
dBy_dy = np.gradient(By, dy, axis=0, edge_order=2)
dBz_dz = np.gradient(Bz, dz, axis=2, edge_order=2)
divB   = dBx_dx + dBy_dy + dBz_dz

# --- Mask: exclude one-cell boundaries & avoid tiny |B| ---
interior = np.zeros_like(Bmag, dtype=bool)
interior[1:-1, 1:-1, 1:-1] = True

# Ignore the weakest 2% to avoid division by ~0 and extreme relative noise
b_thresh = np.nanpercentile(Bmag, 2)
strong_B = Bmag > max(b_thresh, 1e-12)

mask = interior & strong_B & np.isfinite(divB) & np.isfinite(Bmag)

# --- Dimensionless solenoidality metric: |divB| * h / |B| ---
rel_div = np.empty_like(Bmag)
rel_div[:] = np.nan
rel_div[mask] = np.abs(divB[mask]) * h / (Bmag[mask] + 1e-12)

# --- Diagnostics ---
p50 = float(np.nanpercentile(rel_div, 50))
p95 = float(np.nanpercentile(rel_div, 95))
p99 = float(np.nanpercentile(rel_div, 99))
meanB = float(np.nanmean(Bmag))

print(
    f"mean|B|={meanB:.3e} | "
    f"median(rel_div)={p50:.2e} | p95={p95:.2e} | p99={p99:.2e} | "
    f"mask_coverage={(np.isfinite(rel_div).sum()/rel_div.size*100):.1f}%"
)

# --- Assertions (dimensionless tolerances) ---
assert meanB > 1e-6, "Mean |B| too small — field may be trivial."
assert p99 < 5e-2,  "Scaled divergence too high in top 1% of interior, strong-|B| voxels."

print("✓ Non-trivial |B| and near-solenoidal numerics validated (dimensionless, interior-only).")

In [ ]:
# Grid convergence (mean |B| on a common spherical shell across resolutions)

import numpy as np

def shell_mean_B_for_grid(n, t=0.0, shell=(0.9, 1.1)):
    """Mean |B| on a near-surface shell for a given grid size n."""
    fld = UranusEntropicField(grid_size=n)
    Bx, By, Bz = fld.magnetic_field_components(t)
    Bmag = np.sqrt(Bx**2 + By**2 + Bz**2)
    R = np.sqrt(fld.X**2 + fld.Y**2 + fld.Z**2)
    mask = (R > shell[0]) & (R < shell[1])
    return float(np.nanmean(Bmag[mask]))

grids = [40, 60, 80]
means = {n: shell_mean_B_for_grid(n) for n in grids}

ref_n = 60
ref_val = means[ref_n]
ratios = {n: (means[n] / ref_val if ref_val != 0 else np.nan) for n in grids}

print("Shell-mean |B| by grid:", means)
print(f"Relative to {ref_n}^3:", ratios)

# Expect ~±20% due to fixed 1-cell smoothing (resolution-dependent in physical units)
TOL = 0.20
assert all(abs(ratios[n] - 1.0) <= TOL for n in grids), \
    f"Shell mean |B| varies >{int(TOL*100)}% across grids — consider refining or parameterizing smoothing."
print(f"✓ Grid convergence (shell mean |B|) within ±{int(TOL*100)}%.")

### Methods — Numerical details

- **Grid:** uniform Cartesian $$x,y,z\in[-2,2]$$ with `grid_size = 50–80` (submission default 50).  
- **Derivatives:** central finite differences with `edge_order=2` for $$\nabla\Phi_s, \;\nabla^2\Phi_s$$ and directional derivatives.  
- **Smoothing:** mild Gaussian filter (`sigma = 1.0`) on $$\mathbf{B}$$ to suppress discretisation noise without altering large-scale geometry.  
- **Near-surface shell:** voxels with $$0.9<r<1.1$$ in Uranus radii.  
- **Temporal analysis (Pred. 1):** $$\langle|\mathbf{B}|\rangle$$ on shell vs. time; dominant period from FFT peak (DC removed).  
- **Latitudinal profile (Pred. 2):** binning of shell voxels by latitude $$\varphi=\arcsin(z/r)$$ in 2.5° bins; average $$|\mathbf{B}|$$.  
- **Perturbation (Pred. 3):** +10% transient boost of the primary entropic-source amplitude for 6 time units; response measured as $$\langle|\mathbf{B}|\rangle$$. Amplitude restored after loop.  
- **Magnetic tilt (Pred. 4):** principal direction of $$\mathbf{B}$$ on shell via weighted eigen-decomposition of unit vectors; angle to spin axis gives predicted tilt.